# 14.3 提示优化 (Prompt Optimization)

自动优化提示以获得更好的模型输出，将提示工程从手工调优提升为系统化的优化过程。

## 提示优化方法总览

| 方法 | 优化空间 | 需要梯度 | 适用模型 | 代表工作 |
|------|----------|----------|----------|----------|
| APE | 离散文本 | 否 | 黑盒API | Zhou et al. |
| Soft Prompt | 连续向量 | 是 | 白盒模型 | Lester et al. |
| Prefix Tuning | 连续前缀 | 是 | 白盒模型 | Li & Liang |
| P-Tuning v2 | 深层前缀 | 是 | 白盒模型 | Liu et al. |
| DSPy | 编程式 | 否 | 黑盒API | Khattab et al. |
| OPRO | LLM优化LLM | 否 | 黑盒API | Yang et al. |

## 1. 自动提示工程（APE）

APE通过生成-评估循环自动搜索最优提示文本，无需访问模型梯度。

### APE流程
1. **生成**：用LLM生成多个候选提示
2. **评估**：在每个候选提示上评估任务性能
3. **选择**：选择得分最高的提示
4. **迭代**：基于最佳提示生成变体，重复评估

### APE优势
- 适用于黑盒API（如GPT-4）
- 找到的提示人类可读
- 可以发现人类想不到的有效提示

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class APEOptimizer:
    def __init__(self, model, eval_fn, n_candidates=10, n_iterations=5):
        self.model = model
        self.eval_fn = eval_fn
        self.n_candidates = n_candidates
        self.n_iterations = n_iterations
        self.prompt_history = []

    def generate_candidates(self, base_prompt_emb, variation_scale=0.3):
        candidates = []
        for _ in range(self.n_candidates):
            noise = variation_scale * torch.randn_like(base_prompt_emb)
            candidate = base_prompt_emb + noise
            candidates.append(candidate)
        return candidates

    def evaluate_prompt(self, prompt_emb, eval_data):
        x, y = eval_data
        with torch.no_grad():
            combined = torch.cat([prompt_emb.expand(x.shape[0], -1, -1),
                                  self.model.embed(x)], dim=1)
            h = self.model.transformer(combined)
            logits = self.model.head(h[:, prompt_emb.shape[1]:])
            preds = logits.argmax(dim=-1)
            accuracy = (preds == y).float().mean().item()
        return accuracy

    def optimize(self, initial_prompt_emb, eval_data):
        best_prompt = initial_prompt_emb
        best_score = self.evaluate_prompt(best_prompt, eval_data)
        self.prompt_history.append({'score': best_score, 'iteration': 0})

        for it in range(self.n_iterations):
            scale = 0.3 * (1 - it / self.n_iterations)
            candidates = self.generate_candidates(best_prompt, variation_scale=scale)

            for cand in candidates:
                score = self.evaluate_prompt(cand, eval_data)
                if score > best_score:
                    best_score = score
                    best_prompt = cand

            self.prompt_history.append({'score': best_score, 'iteration': it + 1})

        return best_prompt, best_score

class PromptModelForAPE(nn.Module):
    def __init__(self, d=64, vocab_size=50):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d)
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d, 2, d * 4, batch_first=True), 1
        )
        self.head = nn.Linear(d, vocab_size)
    def forward(self, x):
        h = self.embed(x)
        h = self.transformer(h)
        return self.head(h)

model = PromptModelForAPE()
eval_fn = lambda p: 0.5
optimizer = APEOptimizer(model, eval_fn, n_candidates=8, n_iterations=10)

initial_prompt = torch.randn(1, 3, 64) * 0.02
eval_x = torch.randint(0, 50, (16, 5))
eval_y = torch.randint(0, 50, (16, 5))

best_prompt, best_score = optimizer.optimize(initial_prompt, (eval_x, eval_y))

print('=== APE (Automatic Prompt Engineer) ===')
print(f'Initial score: {optimizer.prompt_history[0]["score"]:.4f}')
print(f'Best score: {best_score:.4f}')
print(f'Improvement: {best_score - optimizer.prompt_history[0]["score"]:.4f}')
print(f'\nOptimization history:')
for entry in optimizer.prompt_history:
    print(f'  Iteration {entry["iteration"]:2d}: score={entry["score"]:.4f}')

print(f'\nKey: APE automatically searches for optimal prompts via generate-evaluate loop.')
print(f'Variation scale decreases over iterations for fine-grained search.')

## 2. 软提示优化（Soft Prompt Tuning）

软提示优化将提示表示为连续向量，通过梯度优化找到最优提示嵌入，可以找到文本提示无法表示的最优解。

### 核心思想
- 文本提示是离散的，搜索空间有限
- 软提示是连续的，搜索空间无限
- 梯度优化可以高效搜索连续空间

### 与Prefix Tuning的区别
- **Soft Prompt Tuning**：只在输入层添加可学习向量
- **Prefix Tuning**：在每层Transformer都添加可学习向量
- **P-Tuning v2**：Prefix Tuning的改进版，适用于各种模型规模

In [ ]:
class SoftPromptTuning(nn.Module):
    def __init__(self, base_model, n_soft_tokens=5, d=64, init_range=0.02):
        super().__init__()
        self.base_model = base_model
        self.n_soft_tokens = n_soft_tokens
        self.soft_prompt = nn.Parameter(torch.randn(n_soft_tokens, d) * init_range)

    def forward(self, x):
        B = x.shape[0]
        txt_embeds = self.base_model.embed(x)
        soft_tokens = self.soft_prompt.unsqueeze(0).expand(B, -1, -1)
        combined = torch.cat([soft_tokens, txt_embeds], dim=1)
        h = self.base_model.transformer(combined)
        return self.base_model.head(h[:, self.n_soft_tokens:])

class PrefixTuning(nn.Module):
    def __init__(self, base_model, n_prefix=5, d=64, n_layers=1):
        super().__init__()
        self.base_model = base_model
        self.n_prefix = n_prefix
        self.prefix_embeddings = nn.ParameterList([
            nn.Parameter(torch.randn(n_prefix, d) * 0.02)
            for _ in range(n_layers)
        ])
        self.prefix_mlp = nn.Sequential(
            nn.Linear(d, d * 2), nn.ReLU(), nn.Linear(d * 2, d)
        )

    def forward(self, x):
        B = x.shape[0]
        txt_embeds = self.base_model.embed(x)
        prefix = self.prefix_mlp(self.prefix_embeddings[0].unsqueeze(0).expand(B, -1, -1))
        combined = torch.cat([prefix, txt_embeds], dim=1)
        h = self.base_model.transformer(combined)
        return self.base_model.head(h[:, self.n_prefix:])

base_model = PromptModelForAPE(d=64, vocab_size=50)

soft_model = SoftPromptTuning(base_model, n_soft_tokens=5, d=64)
prefix_model = PrefixTuning(base_model, n_prefix=5, d=64, n_layers=1)

for name, param in soft_model.named_parameters():
    if 'soft_prompt' not in name:
        param.requires_grad = False

for name, param in prefix_model.named_parameters():
    if 'prefix' not in name:
        param.requires_grad = False

x = torch.randint(0, 50, (4, 5))
y = torch.randint(0, 50, (4, 5))

opt_soft = torch.optim.AdamW(soft_model.parameters(), lr=1e-2)
opt_prefix = torch.optim.AdamW(prefix_model.parameters(), lr=1e-2)

print('=== Soft Prompt Tuning vs Prefix Tuning ===')
soft_params = sum(p.numel() for p in soft_model.parameters() if p.requires_grad)
prefix_params = sum(p.numel() for p in prefix_model.parameters() if p.requires_grad)
print(f'Soft prompt trainable params: {soft_params}')
print(f'Prefix tuning trainable params: {prefix_params}')

for epoch in range(20):
    logits_soft = soft_model(x)
    loss_soft = F.cross_entropy(logits_soft.reshape(-1, 50), y.reshape(-1))
    opt_soft.zero_grad()
    loss_soft.backward()
    opt_soft.step()

    logits_prefix = prefix_model(x)
    loss_prefix = F.cross_entropy(logits_prefix.reshape(-1, 50), y.reshape(-1))
    opt_prefix.zero_grad()
    loss_prefix.backward()
    opt_prefix.step()

    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1:3d}: Soft={loss_soft.item():.4f}, Prefix={loss_prefix.item():.4f}')

print(f'\nKey: Soft Prompt only adds input-level vectors.')
print(f'Prefix Tuning adds per-layer vectors via MLP, more expressive but more params.')

## 3. OPRO（LLM优化LLM）

OPRO用LLM来优化LLM的提示，通过元优化循环让LLM自己发现更好的提示。

### OPRO流程
1. 初始化一组提示及其得分
2. 将(提示, 得分)对作为上下文，让LLM生成新提示
3. 评估新提示的得分
4. 将新(提示, 得分)加入上下文
5. 重复直到收敛

### OPRO优势
- 完全黑盒优化，不需要梯度
- 生成的提示人类可读
- 可以优化任何LLM的提示

In [ ]:
class OPROOptimizer:
    def __init__(self, model, eval_fn, n_init=3, n_iterations=10):
        self.model = model
        self.eval_fn = eval_fn
        self.n_init = n_init
        self.n_iterations = n_iterations
        self.prompt_scores = []

    def initialize_prompts(self, d=64, prompt_len=3):
        for _ in range(self.n_init):
            prompt_emb = torch.randn(1, prompt_len, d) * 0.02
            score = self.eval_fn(prompt_emb)
            self.prompt_scores.append((prompt_emb, score))
        self.prompt_scores.sort(key=lambda x: x[1], reverse=True)

    def generate_new_prompt(self, top_k=3, noise_scale=0.1):
        top_prompts = self.prompt_scores[:top_k]
        weights = torch.tensor([s for _, s in top_prompts])
        weights = F.softmax(weights, dim=0)

        weighted_sum = torch.zeros_like(top_prompts[0][0])
        for (p, _), w in zip(top_prompts, weights):
            weighted_sum += w * p

        noise = noise_scale * torch.randn_like(weighted_sum)
        new_prompt = weighted_sum + noise
        return new_prompt

    def optimize(self, d=64, prompt_len=3):
        self.initialize_prompts(d, prompt_len)
        best_score = self.prompt_scores[0][1]
        best_prompt = self.prompt_scores[0][0]
        history = [{'iteration': 0, 'best_score': best_score}]

        for it in range(self.n_iterations):
            noise_scale = 0.2 * (1 - it / self.n_iterations)
            new_prompt = self.generate_new_prompt(top_k=3, noise_scale=noise_scale)
            new_score = self.eval_fn(new_prompt)
            self.prompt_scores.append((new_prompt, new_score))
            self.prompt_scores.sort(key=lambda x: x[1], reverse=True)

            if new_score > best_score:
                best_score = new_score
                best_prompt = new_prompt

            history.append({'iteration': it + 1, 'best_score': best_score})

        return best_prompt, best_score, history

def simple_eval(prompt_emb):
    return torch.sigmoid(prompt_emb.mean()).item() + 0.3 * torch.randn(1).item()

opro = OPROOptimizer(model=None, eval_fn=simple_eval, n_init=5, n_iterations=15)
best_prompt, best_score, history = opro.optimize(d=64, prompt_len=3)

print('=== OPRO (LLM Optimizes LLM Prompts) ===')
for entry in history:
    print(f'  Iteration {entry["iteration"]:2d}: best_score={entry["best_score"]:.4f}')

print(f'\nFinal best score: {best_score:.4f}')
print(f'\nKey: OPRO uses LLM to generate better prompts based on past performance.')
print(f'Noise decreases over iterations for fine-grained optimization.')

## 4. 提示鲁棒性

提示鲁棒性指提示对输入扰动、格式变化和翻译的稳定性。不鲁棒的提示在实际部署中会带来严重问题。

### 鲁棒性问题
- **格式敏感**：换行、空格、标点变化导致输出剧变
- **顺序敏感**：few-shot示例顺序影响结果
- **同义替换**：换一种说法效果差异大
- **对抗性输入**：恶意构造的输入绕过提示约束

### 提高鲁棒性的方法
1. **Ensemble提示**：使用多个提示变体，取平均或投票
2. **一致性约束**：要求模型对等价输入给出一致输出
3. **输入规范化**：预处理输入，消除格式差异
4. **对抗训练**：在对抗性输入上测试和优化提示

In [ ]:
class PromptRobustnessEvaluator:
    def __init__(self, model, d=64, vocab_size=50):
        self.model = model
        self.d = d
        self.vocab_size = vocab_size

    def format_perturbation(self, x, perturbation_type='noise', scale=0.1):
        emb = self.model.embed(x)
        if perturbation_type == 'noise':
            perturbed = emb + scale * torch.randn_like(emb)
        elif perturbation_type == 'dropout':
            mask = torch.bernoulli(torch.ones_like(emb) * (1 - scale))
            perturbed = emb * mask
        elif perturbation_type == 'shuffle':
            perm = torch.randperm(x.shape[1])
            perturbed = self.model.embed(x[:, perm])
        else:
            perturbed = emb
        return perturbed

    def evaluate_robustness(self, x, n_perturbations=10, scale=0.1):
        with torch.no_grad():
            original_emb = self.model.embed(x)
            original_h = self.model.transformer(original_emb)
            original_out = self.model.head(original_h)
            original_pred = original_out.argmax(dim=-1)

        consistency_scores = {}
        for p_type in ['noise', 'dropout', 'shuffle']:
            consistent = 0
            for _ in range(n_perturbations):
                perturbed_emb = self.format_perturbation(x, p_type, scale)
                with torch.no_grad():
                    h = self.model.transformer(perturbed_emb)
                    out = self.model.head(h)
                    pred = out.argmax(dim=-1)
                    consistent += (pred == original_pred).float().mean().item()
            consistency_scores[p_type] = consistent / n_perturbations

        return consistency_scores

    def ensemble_predict(self, x, n_variants=5, scale=0.1):
        all_preds = []
        for _ in range(n_variants):
            perturbed_emb = self.format_perturbation(x, 'noise', scale)
            with torch.no_grad():
                h = self.model.transformer(perturbed_emb)
                out = self.model.head(h)
                probs = F.softmax(out, dim=-1)
                all_preds.append(probs)
        avg_probs = torch.stack(all_preds).mean(dim=0)
        return avg_probs.argmax(dim=-1), avg_probs

model = PromptModelForAPE(d=64, vocab_size=50)
evaluator = PromptRobustnessEvaluator(model, d=64, vocab_size=50)

x = torch.randint(0, 50, (4, 5))
robustness = evaluator.evaluate_robustness(x, n_perturbations=10, scale=0.1)

print('=== Prompt Robustness ===')
print(f'Consistency under perturbations (scale=0.1):')
for p_type, score in robustness.items():
    print(f'  {p_type:10s}: {score:.2%} consistent')

ensemble_pred, ensemble_probs = evaluator.ensemble_predict(x, n_variants=5)
print(f'\nEnsemble prediction (5 variants): {ensemble_pred.tolist()}')
print(f'Ensemble confidence: {ensemble_probs.max(dim=-1).values.mean():.4f}')

print(f'\nKey: Robust prompts should maintain consistency under perturbations.')
print(f'Ensemble prediction is more robust than single prediction.')

## 5. 提示优化工业实践

### 方法选择
| 场景 | 推荐方法 | 原因 |
|------|----------|------|
| 黑盒API | APE / OPRO | 不需要梯度 |
| 白盒模型 | Soft Prompt / Prefix Tuning | 梯度优化更高效 |
| 少样本 | Prefix Tuning | 每层前缀提供更强表达 |
| 追求可解释 | APE | 生成的提示人类可读 |
| 追求性能 | Soft Prompt + 模型微调 | 连续空间搜索更优 |

### 提示优化最佳实践
1. **先手动设计，再自动优化**：手动设计提供好的初始化
2. **评估集要代表性**：优化提示在评估集上过拟合是常见问题
3. **多指标评估**：不只看准确率，还要看鲁棒性、延迟、成本
4. **版本管理**：提示变体需要版本控制和A/B测试
5. **定期重新优化**：模型更新后，旧提示可能不再最优

### 提示优化的局限
- **过拟合风险**：优化到评估集上，泛化性差
- **不可解释**：软提示无法解读为自然语言
- **模型依赖**：为某模型优化的提示不一定适用于其他模型
- **成本**：APE和OPRO需要大量API调用

### 未来方向
- **DSPy**：将提示工程变为编程问题，自动优化提示管道
- **多模态提示**：图像+文本的联合提示优化
- **提示压缩**：在保持效果的同时缩短提示长度
- **个性化提示**：为不同用户自动优化个性化提示